In [1]:
import re
import numpy as np
import pandas as pd
from src.recovery_model import RecoveryModel

# pd.set_option("multi_sparse", False)

In [2]:
# Select a folder for the data to be used
folder = "test_2"  # test_1  test_2  Toy_WEEE_v2
layer_0 = "flow"
layer_1 = "product"
layer_2 = "component"
layer_3 = "material"
layer_4 = "element"

In [28]:
layer_names = (layer_0, layer_1, layer_2, layer_3, layer_4)
all_symbols = {layer_1: "P*", layer_2: "C*", layer_3: "M*", layer_4: "E*"}

metadata = {
    "path": f"data/{folder}/",
    "filename": "metadata.csv",
}
composition = {
    "path": f"data/{folder}/",
    "filename": "composition.csv",
    "mapper": {
        "Stock/Flow ID": layer_0,
        "Layer 1": layer_1,
        "Layer 2": layer_2,
        "Layer 3": layer_3,
        "Layer 4": layer_4,
        "Value": "data",
        "Year": "year",
        "Scenario": "scenario",
        "Location": "region",
        "parameterCode": "parameterCode",
        "UoM": "unit",
    },
    "parameterCode": {
        layer_2: "c-p",
        layer_3: "m-c",
        layer_4: "e-m",
    },
}

inputs = {
    "path": f"data/{folder}/",
    "filename": "inputs.csv",
    "mapper": {
        "Stock/Flow ID": layer_0,
        "Substance_main_parent": layer_1,
        "Value": "data",
        "Year": "year",
        "Unit": "unit",
    },
}

tcs = {
    "path": f"data/{folder}/",
    "filename": "TCs.csv",
    "mapper": {
        # inflows / outflows
        "input_flow": "inflows",
        "output_flow": "outflows",
        # levels
        "input_layer": "input_layer1_level",
        "input_sub_layer": "input_layer2_level",
        "output_layer": "output_layer1_level",
        # keys
        "input_layer_key": "input_layer1_key",
        "input_sub_layer_key": "input_layer2_key",
        "output_target_key": "output_layer1_key",
        # other columns
        "value": "data",
        "Year": "year",
    },
    "all_symbols": {
        layer_1: "P*",
        layer_2: "C*",
        layer_3: "M*",
        layer_4: "E*",
    },
}

---

# Composition file


In [33]:
composition["filename"] = "composition.csv"
# composition["filename"] = "composition_with_duplicates.csv"  # only for Toy_WEEE_v2
composition["filename"] = "composition_imbalanced.csv"  # only for Toy_WEEE_v2


df_comp = (
    pd.read_csv(composition["path"] + composition["filename"])
    .rename(columns=composition["mapper"])
    .reset_index(drop=False)
)
df_comp["index"] += 2

## Duplicate rows


In [34]:
# check if there are duplicated rows
duplicates = df_comp[list(layer_names)].duplicated(keep=False)
if duplicates.any():
    display(df_comp.loc[duplicates, :])
    raise ValueError("Duplicated rows in composition")

## Level bypassing


In [35]:
# check if a level is being bypassed incorrectly
# e.g. [F1, P1, C1, / , E1]
pattern = re.compile(r"^0*1*0*$")
arr = df_comp[list(layer_names)].notna().to_numpy(dtype=np.int8)
arr = np.array(["".join(map(str, row)) for row in arr])
incomplete_lvl = np.array([False if pattern.match(s) else True for s in arr])
if incomplete_lvl.any():
    display(df_comp.loc[incomplete_lvl, :].astype(str).replace("nan", "?"))
    raise ValueError("Missing level information in composition")

## Mass balance of composition

Entities that are not mass balanced are saved in a separate csv file `src/consolidation`.


In [36]:
for lvl, layer in enumerate(layer_names[:-1]):
    # For each layer, we ignore the rows that represent sub-levels, for example:
    # For example when looking at component level, we remove the following:
    # [F1, P1, C1, M1, / ] = a        [F1, P1, C1, M1, / ] = a
    # [F1, P1, C1, M2, / ] = b        [F1, P1, C1, M2, / ] = b
    # [F1, P1, C1, / , / ] = c   -->  [------------------]----
    # [F1, P1, C1, M3, E1] = e        [------------------]----
    # The last row represents the share of E1 in [F1, P1, C1, M3]
    # And the row before that represents the share of C1 in [F1, P1]
    # In practice this means that, we only keeps the rows where:
    #  - the next layer is NOT empty
    #  - AND all the following layers are empty
    rows = (df_comp.loc[:, layer_names[lvl + 1]].notna()) & (df_comp.loc[:, list(layer_names[lvl + 2 :])].isna()).all(
        axis=1
    )
    columns = list(layer_names[: lvl + 2]) + ["data"]
    # then we pivot the tablet, and we check the sum accross the columns
    # (within a certain tolerance because of floating imprecision)
    # [F1, P1, C1, M1, / ] = a                      | M1 | M2 |
    # [F1, P1, C1, M2, / ] = b   --> [F1, P1, C1] = | a  | b  | == 1 ?
    df_lvl = df_comp.loc[rows, columns].pivot(
        index=layer_names[: lvl + 1],
        columns=layer_names[lvl + 1],
        values="data",
    )
    df_lvl["total"] = df_lvl.sum(axis=1)
    mask = ~np.isclose(df_lvl["total"], 1, rtol=1e-5, atol=1e-5)
    if mask.any():
        print(f"Mass imbalance in {layer}")
        df_lvl.loc[mask].to_csv(f"consolidation/{folder}_composition_mass_balance_{layer}.csv")

Mass imbalance in flow
Mass imbalance in product
Mass imbalance in component
Mass imbalance in material


---

# TCs file


## Conflicting flows information

Check if any flow has multiple processes as source / sink


In [8]:
# tcs["filename"] = "TCs_original.csv"
tcs["filename"] = "TCs.csv"


df_tcs = pd.read_csv(tcs["path"] + tcs["filename"]).rename(columns=tcs["mapper"]).reset_index(drop=False)
from_ = df_tcs[["outflows", "process"]].rename(columns={"process": "from", "outflows": "flows"}).drop_duplicates()
to_ = df_tcs[["process", "inflows"]].rename(columns={"process": "to", "inflows": "flows"}).drop_duplicates()
groupby_from = from_.groupby(["flows"])["from"].agg(["unique", "count"]).rename({"unique": "process"}, axis=1)
groupby_to = to_.groupby(["flows"])["to"].agg(["unique", "count"]).rename({"unique": "process"}, axis=1)
df_tcs = pd.concat({"from": groupby_from, "to": groupby_to}, axis=1)
cols = pd.IndexSlice
non_uniques = (df_tcs.loc[:, cols[:, "count"]].fillna(0) > 1).any(axis=1)

if non_uniques.any():
    display(df_tcs.loc[non_uniques, cols[:, "process"]])
    raise ValueError("Multiple from/to processes for the same flow")

## Conflicting TC values

To find if there's any conflicting rows, run the model with the option `save_duplicates` enabled. This will be saved in the folder `src/consolidation`


In [9]:
model = RecoveryModel(
    name=folder,
    metadata=metadata,
    composition=composition,
    inputs=inputs,
    tcs=tcs,
    layer_names=layer_names,
    save_intermediary_steps=False,
    save_duplicates=True,  # Make sure to save_duplicates
)

In [10]:
# check if there are duplicated rows
file = f"consolidation/{folder}_TCs_duplicates.csv"
try:
    duplicates = pd.read_csv(file, header=[0, 1], index_col=0).drop(
        columns=[("info", "process"), ("info", "technology")]
    )
    display(duplicates)
except FileNotFoundError:
    print("No duplicates found in TCs")

,info,outflow,outflow,outflow,outflow,outflow,inflow,inflow,inflow,inflow,inflow,info,info
,original row,flow,product,component,material,element,flow,product,component,material,element,data,priority
15,15,6,-1,0,0,0,3,-1,0,0,0,0.51,2211
17,17,6,0,0,0,0,4,0,0,0,0,0.40,2211
17,17,6,0,-1,0,0,4,0,-1,0,0,0.40,2211
17,17,6,-1,2,0,0,4,-1,2,0,0,0.40,2211
17,17,6,-1,1,0,0,4,-1,1,0,0,0.40,2211
...,...,...,...,...,...,...,...,...,...,...,...,...,...
13,13,6,-1,2,0,0,3,-1,2,0,0,0.59,2211
13,13,6,0,-1,0,0,3,0,-1,0,0,0.59,2211
13,13,6,0,0,0,0,3,0,0,0,0,0.59,2211


Only run the cell below if there was duplicates


In [11]:
# temporary rename columns for easier processing
cols = list(duplicates.columns)
duplicates.columns = list(range(len(duplicates.columns)))
duplicates[0] = duplicates[0] + 2  # for exact match in excel

# get what the conflict values (for tc) are
s = duplicates.groupby(list(range(1, 11)))[11].unique().to_frame()
s[("info", "values_count")] = s[11].map(len)

# get what the conflict priorities (for tc) are
s2 = duplicates.groupby(list(range(1, 11)))[12].unique().to_frame()
s2[("info", "priority_count")] = s2[12].map(len)
s = pd.concat([s, s2], axis=1)
del s2

# get what original rows are concerned (in the tcs csv file)
s2 = duplicates.groupby(list(range(1, 11)))[0].unique().to_frame()
s = pd.concat([s, s2], axis=1)
del s2

# rename
s = s.reset_index().rename(columns={i: v for i, v in enumerate(cols)})
s.columns = pd.MultiIndex.from_tuples(s.columns)
s = model.decode_label(s)
s.to_csv(f"consolidation/{folder}_TCs_collision.csv")
s.fillna("")

,outflow,outflow,outflow,outflow,outflow,inflow,inflow,inflow,inflow,inflow,info,info,info,info,info
,flow,product,component,material,element,flow,product,component,material,element,data,values_count,priority,priority_count,original row
0,F7,,,M1,E1,F4,,,M1,E1,"[0.51, 0.73, 0.24, 0.59]",4,[2211],1,"[17, 14, 16, 15]"
1,F7,,,M1,E1,F5,,,M1,E1,"[0.4, 0.36, 0.52, 0.44]",4,[2211],1,"[19, 18, 20, 21]"
2,F7,,C1,M1,E1,F4,,C1,M1,E1,"[0.51, 0.73, 0.24, 0.59]",4,[2211],1,"[17, 14, 16, 15]"
3,F7,,C1,M1,E1,F5,,C1,M1,E1,"[0.4, 0.36, 0.44, 0.52]",4,[2211],1,"[19, 18, 21, 20]"
4,F7,,C2,M1,E1,F4,,C2,M1,E1,"[0.51, 0.73, 0.24, 0.59]",4,[2211],1,"[17, 14, 16, 15]"
5,F7,,C2,M1,E1,F5,,C2,M1,E1,"[0.4, 0.36, 0.44, 0.52]",4,[2211],1,"[19, 18, 21, 20]"
6,F7,,C3,M1,E1,F4,,C3,M1,E1,"[0.51, 0.73, 0.24, 0.59]",4,[2211],1,"[17, 14, 16, 15]"
7,F7,,C3,M1,E1,F5,,C3,M1,E1,"[0.4, 0.36, 0.44, 0.52]",4,[2211],1,"[19, 18, 21, 20]"
8,F7,P1,,M1,E1,F4,P1,,M1,E1,"[0.51, 0.73, 0.24, 0.59]",4,[2211],1,"[17, 14, 16, 15]"


---

# Solver


## Mass balance


In [13]:
# First let's make sure we ran the solver
# model.solve(aggregate=True, pivot=True).fillna("")

mass_balance = pd.read_csv(f"consolidation/{folder}_solution_mass_balance.csv", index_col=0)
mass_balance.round(2).fillna("").head(15)

,product,component,material,element,process,F1,F2,F3,F4,F5,F6,F7,F8,mass_balance
0,P1,,,,T1,1000.00,,,,,,,,1000.00
1,P1,C1,,,T1,250.00,-62.5,,,,,,,187.50
2,P1,C1,M1,,T1,130.00,-32.5,,,0.0,,,,97.50
3,P1,C1,M1,E1,T1,91.00,-22.75,,,0.0,,,0.0,68.25
4,P1,C1,M1,E2,T1,39.00,-9.75,,,0.0,,,,29.25
5,P1,C1,M2,,T1,120.00,-30.0,,,0.0,,,,90.00
6,P1,C1,M2,E1,T1,117.60,-29.4,,,0.0,,,,88.20
7,P1,C1,M2,E2,T1,2.40,-0.6,,,0.0,,,0.0,1.80
8,P1,C2,,,T1,590.00,-27.76,-26.86,0.0,,11.24,,,546.62
9,P1,C2,M1,,T1,389.40,-18.32,-17.73,0.0,0.0,7.42,0.0,,360.77


In [23]:
mass_balance.iloc[np.r_[0:6, 44:49, 93:97, 72:77]].round(2).fillna("")

,product,component,material,element,process,F1,F2,F3,F4,F5,F6,F7,F8,mass_balance
0,P1,,,,T1,1000.0,,,,,,,,1000.00
1,P1,C1,,,T1,250.0,-62.5,,,,,,,187.50
2,P1,C1,M1,,T1,130.0,-32.5,,,0.0,,,,97.50
3,P1,C1,M1,E1,T1,91.0,-22.75,,,0.0,,,0.0,68.25
4,P1,C1,M1,E2,T1,39.0,-9.75,,,0.0,,,,29.25
5,P1,C1,M2,,T1,120.0,-30.0,,,0.0,,,,90.00
44,P1,C1,,,T2,0.0,62.5,,,,,,,62.50
45,P1,C1,M1,,T2,0.0,32.5,,,-12.68,,,,19.82
46,P1,C1,M1,E1,T2,0.0,22.75,,,-8.87,,,0.0,13.88
47,P1,C1,M1,E2,T2,0.0,9.75,,,-3.8,,,,5.95


In [25]:
process = "T4"
mask = mass_balance["process"] == process
mass_balance.round(2).fillna("").loc[mask, :]

,product,component,material,element,process,F1,F2,F3,F4,F5,F6,F7,F8,mass_balance
72,P1,C2,,,T4,0.0,0.0,26.86,5.55,,-11.24,,,21.17
73,P1,C2,M1,,T4,0.0,0.0,17.73,3.66,0.0,-7.42,-2.08,,11.89
74,P1,C2,M1,E1,T4,0.0,0.0,4.96,1.03,0.0,-2.08,-0.58,0.0,3.33
75,P1,C2,M1,E2,T4,0.0,0.0,12.76,2.64,0.0,-5.34,-1.5,,8.56
76,P1,C2,M2,,T4,0.0,0.0,9.13,1.89,0.0,-3.82,-2.04,,5.16
77,P1,C2,M2,E1,T4,0.0,0.0,0.73,0.15,0.0,-0.31,-0.16,,0.41
78,P1,C2,M2,E2,T4,0.0,0.0,8.4,1.74,0.0,-3.52,-1.88,0.0,4.74
79,P1,C3,,,T4,0.0,,13.04,,,-1.83,,,11.21
80,P1,C3,M1,,T4,0.0,,6.39,,,-0.89,-0.83,,4.66
81,P1,C3,M1,E1,T4,0.0,,2.04,,,-0.29,-0.27,0.0,1.49


In [10]:
sol = pd.read_csv("results/test_2_solution_agg=False_pivot=True.csv", index_col=0)
sol.round(2).fillna("")

,product,component,material,element,F1,F2,F3,F4,F5,F6,F7,F8
0,P1,,,,1000.00,,,,,,,
1,P1,C1,,,250.00,62.5,,,,,,
2,P1,C1,M1,,130.00,32.5,,,12.68,,,
3,P1,C1,M1,E1,91.00,22.75,,,8.87,,,1.6
4,P1,C1,M1,E2,39.00,9.75,,,3.8,,,
...,...,...,...,...,...,...,...,...,...,...,...,...
39,P2,C3,M1,E1,85.75,,,,,,,
40,P2,C3,M1,E2,36.75,,,,,,,
41,P2,C3,M2,,227.50,,,,,,,
42,P2,C3,M2,E1,65.97,,,,,,,
